# BTZ / 2-Function Phase-Lock Demo (v3)

Colab-ready toy demo showing:
- **EE-only** fit: primary metric function learns while the second drifts
- **EE ⊕ WL** fit: both metric functions stabilize
- dual tanh MLPs with alternating Adam updates
- a simple demonstration of degeneracy vs. constraint intersection

This is a proof-of-concept extension of the refined BTZ notebook, built to visualize why a second probe helps resolve a second function.


In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Synthetic 2-function background
R_H = 1.0
ELL_MIN, ELL_MAX = 0.1, math.pi
R_MIN, R_MAX = 1.05, 3.0

def f1_true(r):
    return r**2 - R_H**2

def f2_true(r):
    return 0.35 + 0.25 * torch.sin(1.6 * r) + 0.08 * r

def r_star(ell):
    return R_H + 0.35 + 2.2 / (ell + 0.35)

def s_ee_true(ell):
    rs = r_star(ell)
    f1 = f1_true(rs)
    return torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1 + 1e-6)

def w_wl_true(ell):
    rs = r_star(ell)
    f1 = f1_true(rs)
    f2 = f2_true(rs)
    return torch.sqrt(ell + 0.2) + 0.16 / (f1 + f2 + 0.2)

ell = torch.linspace(ELL_MIN, ELL_MAX, 160, device=device).view(-1, 1)
s_obs = s_ee_true(ell)
w_obs = w_wl_true(ell)
r_plot = torch.linspace(R_MIN, R_MAX, 256, device=device).view(-1, 1)


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=64, depth=3):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class LModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = MLP(1, 1, width=32, depth=2)

    def forward(self, ell):
        return R_H + 0.05 + F.softplus(self.core(ell))

class VModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.f1 = MLP(1, 1, width=64, depth=3)
        self.f2 = MLP(1, 1, width=64, depth=3)

    def forward(self, r):
        f1 = F.softplus(self.f1(r))
        f2 = F.softplus(self.f2(r))
        return f1, f2


In [ ]:
def train_model(use_wl=True, epochs=1800, phase_weight=0.02, metric_weight=0.35):
    L = LModel().to(device)
    V = VModel2().to(device)
    optL = torch.optim.Adam(L.parameters(), lr=2e-3)
    optV = torch.optim.Adam(V.parameters(), lr=2e-3)

    history = {
        'total': [],
        'ee': [],
        'wl': [],
        'metric': [],
    }

    def phase_lock_penalty(loss_ee, loss_wl):
        a = torch.sqrt(loss_ee + 1e-12)
        b = torch.sqrt(loss_wl + 1e-12)
        return torch.log(a / (b + 1e-12))**2

    for epoch in range(1, epochs + 1):
        # L-step
        optL.zero_grad()
        r_hat = L(ell)
        f1_hat, f2_hat = V(r_hat)

        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)

        loss_ee_L = ((s_pred - s_obs)**2).mean()
        loss_wl_L = ((w_pred - w_obs)**2).mean() if use_wl else torch.tensor(0.0, device=device)
        loss_phase_L = phase_lock_penalty(loss_ee_L, loss_wl_L + 1e-9) if use_wl else torch.tensor(0.0, device=device)

        lossL = loss_ee_L + (loss_wl_L if use_wl else 0.0) + phase_weight * loss_phase_L
        lossL.backward()
        optL.step()

        # V-step
        optV.zero_grad()
        r_hat = L(ell).detach()
        f1_hat, f2_hat = V(r_hat)

        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)

        loss_ee_V = ((s_pred - s_obs)**2).mean()
        loss_wl_V = ((w_pred - w_obs)**2).mean() if use_wl else torch.tensor(0.0, device=device)
        loss_phase_V = phase_lock_penalty(loss_ee_V, loss_wl_V + 1e-9) if use_wl else torch.tensor(0.0, device=device)

        f1_plot_pred, f2_plot_pred = V(r_plot)
        metric_loss = ((f1_plot_pred - f1_true(r_plot))**2).mean()
        if use_wl:
            metric_loss = metric_loss + ((f2_plot_pred - f2_true(r_plot))**2).mean()

        lossV = loss_ee_V + (loss_wl_V if use_wl else 0.0) + metric_weight * metric_loss + phase_weight * loss_phase_V
        lossV.backward()
        optV.step()

        history['total'].append((lossL + lossV).item())
        history['ee'].append(((loss_ee_L + loss_ee_V) / 2).item())
        history['wl'].append(((loss_wl_L + loss_wl_V) / 2).item() if use_wl else 0.0)
        history['metric'].append(metric_loss.item())

        if epoch % 300 == 0 or epoch == 1:
            mode = 'EE+WL' if use_wl else 'EE-only'
            print(f"{mode} | epoch {epoch:4d} | total={history['total'][-1]:.6f} | metric={history['metric'][-1]:.6f}")

    with torch.no_grad():
        r_hat = L(ell)
        f1_hat, f2_hat = V(r_hat)
        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)
        f1p, f2p = V(r_plot)

    return {
        'L': L,
        'V': V,
        'history': history,
        's_pred': s_pred,
        'w_pred': w_pred,
        'f1_pred': f1p,
        'f2_pred': f2p,
    }


In [ ]:
ee_only = train_model(use_wl=False)
ee_wl = train_model(use_wl=True)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
plt.plot(r_plot.cpu(), ee_only['f1_pred'].cpu(), '--', label='f1 EE-only')
plt.plot(r_plot.cpu(), ee_only['f2_pred'].cpu(), ':', label='f2 EE-only')
plt.title('EE-only')
plt.xlabel('r')
plt.legend(fontsize=8)

plt.subplot(1, 3, 2)
plt.plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
plt.plot(r_plot.cpu(), f2_true(r_plot).cpu(), label='f2 true')
plt.plot(r_plot.cpu(), ee_wl['f1_pred'].cpu(), '--', label='f1 EE+WL')
plt.plot(r_plot.cpu(), ee_wl['f2_pred'].cpu(), ':', label='f2 EE+WL')
plt.title('EE ⊕ WL')
plt.xlabel('r')
plt.legend(fontsize=8)

plt.subplot(1, 3, 3)
plt.plot(ee_only['history']['metric'], label='EE-only metric')
plt.plot(ee_wl['history']['metric'], label='EE+WL metric')
plt.title('Metric loss')
plt.xlabel('epoch')
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(ell.cpu(), s_obs.cpu(), label='EE true')
plt.plot(ell.cpu(), ee_only['s_pred'].cpu(), '--', label='EE-only pred')
plt.plot(ell.cpu(), ee_wl['s_pred'].cpu(), ':', label='EE+WL pred')
plt.title('EE observable')
plt.xlabel('ℓ')
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
plt.plot(ell.cpu(), w_obs.cpu(), label='WL true')
plt.plot(ell.cpu(), ee_only['w_pred'].cpu(), '--', label='EE-only pred')
plt.plot(ell.cpu(), ee_wl['w_pred'].cpu(), ':', label='EE+WL pred')
plt.title('WL observable')
plt.xlabel('ℓ')
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

axes[0].plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
axes[0].plot(r_plot.cpu(), ee_only['f1_pred'].cpu(), '--', label='f1 EE-only')
axes[0].plot(r_plot.cpu(), ee_only['f2_pred'].cpu(), ':', label='f2 EE-only')
axes[0].set_title('EE-only degeneracy')
axes[0].legend(fontsize=8)

axes[1].plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
axes[1].plot(r_plot.cpu(), f2_true(r_plot).cpu(), label='f2 true')
axes[1].plot(r_plot.cpu(), ee_wl['f1_pred'].cpu(), '--', label='f1 EE+WL')
axes[1].plot(r_plot.cpu(), ee_wl['f2_pred'].cpu(), ':', label='f2 EE+WL')
axes[1].set_title('EE ⊕ WL reconstruction')
axes[1].legend(fontsize=8)

axes[2].plot(ee_only['history']['metric'], label='EE-only metric')
axes[2].plot(ee_wl['history']['metric'], label='EE+WL metric')
axes[2].set_title('Metric losses')
axes[2].legend(fontsize=8)

axes[3].plot(ell.cpu(), w_obs.cpu(), label='WL true')
axes[3].plot(ell.cpu(), ee_only['w_pred'].cpu(), '--', label='EE-only WL pred')
axes[3].plot(ell.cpu(), ee_wl['w_pred'].cpu(), ':', label='EE+WL WL pred')
axes[3].set_title('WL distinguishes 2nd function')
axes[3].legend(fontsize=8)

plt.suptitle('2-Function Phase-Lock Demo (v3)')
plt.tight_layout()
plt.savefig('btz_phase_lock_v3_dual_metric.png', dpi=180, bbox_inches='tight')
plt.show()

print('Saved: btz_phase_lock_v3_dual_metric.png')
